#langkah 1: iampor pustaka dan dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE


In [2]:
# 1. Membuat dataset tidak seimbang (95:5)
X, y = make_classification(n_samples=1000, n_features=2, n_redundant=0,
n_clusters_per_class=1, weights=[0.95],

flip_y=0, random_state=42)

# Menampilkan distribusi kelas awal
df = pd.DataFrame(X, columns=['Fitur_1', 'Fitur_2'])
df['Target'] = y
print("Distribusi Kelas Asli:\n", df['Target'].value_counts())

Distribusi Kelas Asli:
 Target
0    950
1     50
Name: count, dtype: int64


#Langkah 2: wksperimen tanpa penanganan (baseline)

In [22]:
# Split data dengan Stratified Sampling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
# Latih model MLP tanpa penanganan imbalance
model_bias = MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=42)
model_bias.fit(X_train, y_train)
# Prediksi dan Evaluasi
y_pred_bias = model_bias.predict(X_test)
print("\n--- Evaluasi Model Tanpa SMOTE ---")
print("Akurasi:", accuracy_score(y_test, y_pred_bias))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_bias))
print("\nClassification Report:\n", classification_report(y_test, y_pred_bias))


--- Evaluasi Model Tanpa SMOTE ---
Akurasi: 0.98
Confusion Matrix:
 [[190   0]
 [  4   6]]

Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99       190
           1       1.00      0.60      0.75        10

    accuracy                           0.98       200
   macro avg       0.99      0.80      0.87       200
weighted avg       0.98      0.98      0.98       200



# Langkah 3: implementasi SMOTE (balancing)

In [4]:
# Inisialisasi SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("\nDistribusi Kelas Setelah SMOTE (Data Training):")
print(pd.Series(y_train_res).value_counts())


Distribusi Kelas Setelah SMOTE (Data Training):
0    760
1    760
Name: count, dtype: int64


#langkah 4: Melatih moel setelah balacing

In [5]:
# Latih model MLP dengan data seimbang
model_balanced = MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=42)
model_balanced.fit(X_train_res, y_train_res)
# Prediksi pada data Test (Data Test tetap asli/tidak di-SMOTE)
y_pred_balanced = model_balanced.predict(X_test)
print("\n--- Evaluasi Model DENGAN SMOTE ---")
print("Akurasi:", accuracy_score(y_test, y_pred_balanced))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_balanced))
print("\nClassification Report:\n", classification_report(y_test, y_pred_balanced))


--- Evaluasi Model DENGAN SMOTE ---
Akurasi: 0.99
Confusion Matrix:
 [[189   1]
 [  1   9]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99       190
           1       0.90      0.90      0.90        10

    accuracy                           0.99       200
   macro avg       0.95      0.95      0.95       200
weighted avg       0.99      0.99      0.99       200



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
